## Purpose of This Notebook

This notebook serves to load data from the manually recorded corrections spreadsheet into tables in the CB database
so that they can be applied to the data loaded from the log files.


In [1]:
# | echo: false

from pathlib import Path

import duckdb
import openpyxl
import pandas as pd
from loguru import logger
from openpyxl.utils.dataframe import dataframe_to_rows

from horse_logic.project import Project

### Setup project & directories

In [ ]:
# | echo: false

project = Project(experiment_type="CB", table_page_length=25, delete_existing_db=False)

EXPERIMENT_TYPE = project.experiment_type

DATA_DIR = project.data_dir
LOGFILES_DIR = project.logfiles_dir
NOTEBOOK_DIR = project.notebooks_dir
OUTPUT_DIR = project.output_dir

### Process the Corrections workbook (lookup log files based on subject (horse) name and date)

In [3]:
CORRECTIONS_WORKBOOK = project.project_dir / "docs" / "from_CH" / "Exp1 Errors.xlsx"
WORKSHEET_NAMES = ["CBHD_Times", "CBCSU_Times"]

In [4]:
# | echo: false

def lookup_cb_log_filename(db_path, horse_name, date):
    try:
        with duckdb.connect(str(db_path)) as con:
            query = """
            SELECT LogFileName
            FROM ExperimentCBs
            WHERE LOWER(SubjectName) = LOWER(?)
              AND CAST(DateTime AS DATE) = ?
            """
            result = con.execute(query, [horse_name, date]).fetchall()
        
        log_file_names = [row[0] for row in result]
        logger.debug(f"Query result for {horse_name} on {date}: {log_file_names}")
        return log_file_names
    except Exception as e:
        logger.error(f"Error in lookup_cb_log_filename: {str(e)}")
        return []

def add_log_filename_to_df(df, db_path):
    def get_log_filename(row):
        horse_name = row["Horse"].lower()
        date = row["Date"].date()
        log_file_names = lookup_cb_log_filename(db_path, horse_name, date)
        return log_file_names[0] if log_file_names else None
    
    df["LogFilename"] = df.apply(get_log_filename, axis=1)
    return df

In [5]:
# | echo: false

def preprocess_corrections_df(df, db_path):
    columns_to_ffill = ["Date", "Session", "Horse"]
    df[columns_to_ffill] = df[columns_to_ffill].ffill()

    df = df.drop(columns=["Unnamed: 9"], errors="ignore")
    object_columns = df.select_dtypes(include=["object"]).columns
    df[object_columns] = df[object_columns].fillna("")
    df["Date"] = pd.to_datetime(df["Date"])

    df = add_log_filename_to_df(df, db_path)

    return df

In [6]:
# | echo: false

def process_corrections_sheet(input_file, sheet_name, output_file, db_path):
    df = pd.read_excel(input_file, sheet_name=sheet_name)
    df = preprocess_corrections_df(df, db_path)
    
    wb = openpyxl.load_workbook(input_file)
    processed_sheet_name = f"{sheet_name}_processed"
    if processed_sheet_name in wb.sheetnames:
        wb.remove(wb[processed_sheet_name])
    ws_processed = wb.create_sheet(processed_sheet_name)

    for r in dataframe_to_rows(df, index=False, header=True):
        ws_processed.append(r)
    wb.save(output_file)
    wb.close()
    
    logger.info(f"Processed data saved to '{output_file}' in sheet '{processed_sheet_name}'")
    return df

In [7]:
# | echo: false

def process_correction_worksheets(project):
    processed_sheets = {}
    for sheet_name in WORKSHEET_NAMES:
        output_file = Path(f'{str(CORRECTIONS_WORKBOOK).replace(".xlsx", f"_{sheet_name}.xlsx")}')
        processed_sheets[sheet_name] = process_corrections_sheet(CORRECTIONS_WORKBOOK, sheet_name, output_file, project.DB_FILE_PATH)
    return processed_sheets

In [ ]:
# | echo: false
# | output: false

processed_sheet = process_correction_worksheets(project)

In [ ]:
processed_sheet["CBHD_Times"]

In [ ]:
processed_sheet["CBCSU_Times"]

### Bringing this all together and putting the data in `DuckDB` 

*NOTE: This is a work in progress - storing the table of corrections and then applying it. This requires finding unqiue logfile name (i.e. experiment) matches prior.*

In [11]:
# | echo: false

# project.create_tables_from_sql_file('create_cb_correction_tables_ddb.sql')

In [12]:
# | echo: false

project.close_db_connection()